# Example Notebook for 3D Chebyshev Interpolation

## Importing packages

In [1]:
import numpy as np
from scipy.fft import dctn

from chebND.clenshaw import clenshaw_1d, clenshaw_2d,clenshaw_3d
from chebND.utils import chebyshev_nodes,prune_trailing_slices_greedy_nd, chebyshev_gauss_lobatto_nodes
from chebND.interp3d import ChebyshevInterpolant3D
from numba import njit

from time import (
    process_time,
    perf_counter,
)


## Parallel Definitions

In [2]:
def renormalize_chebyshev_coefficients(out):
    """
    Renormalize a 3D array of Chebyshev coefficients.

    Parameters:
        out (np.ndarray): 3D array of shape (x_num, y_num, z_num)

    Returns:
        np.ndarray: Renormalized 3D array of same shape.
    """
    x_num, y_num, z_num = out.shape

    # Factor arrays: 2 for index 0, 1 elsewhere
    fx = np.ones(x_num)
    fx[0] = 2

    fy = np.ones(y_num)
    fy[0] = 2

    fz = np.ones(z_num)
    fz[0] = 2

    # Compute 3D scale array
    scale = np.einsum('i,j,k->ijk', fx, fy, fz)

    # Normalize
    normalized = out / (scale * x_num * y_num * z_num)

    return normalized

In [3]:
def compute_chebyshev_coefficients_3d(f_values, Nx, Ny, Nz):
    """
    Compute Chebyshev coefficients for a 3D array of function values using DCT.
    Assumes values sampled at Chebyshev nodes on a 3D tensor grid.
    """

    coeffs = dctn(f_values, type=2, norm=None)
    
    #coeffs *= 1 / ((Nx) * (Ny)* (Nz))
    #coeffs[0, 0,:] *= 0.25
    #coeffs[0, :,0] *= 0.25
    #coeffs[:, 0,0] *= 0.25

    coeffs = renormalize_chebyshev_coefficients(coeffs)
    
    return coeffs

In [4]:
def compute_chebyshev_coefficients_3d_GL(f_values, Nx, Ny, Nz):
    """
    Compute Chebyshev coefficients for a 3D array of function values using DCT.
    Assumes values sampled at Chebyshev nodes on a 3D tensor grid.
    """

    coeffs = dctn(f_values, type=1, norm=None)
    
    #coeffs *= 1 / ((Nx-1) * (Ny-1)* (Nz-1))

    #coeffs[0,:,:] *= 0.5
    #coeffs[:,0,:] *= 0.5
    #coeffs[:,:,0] *= 0.5
    #coeffs[-1,:,:] *= 0.5
    #coeffs[:,-1,:] *= 0.5
    #coeffs[:,:,-1] *= 0.5

    coeffs = renormalise_nd_extrema(coeffs)
    
    return coeffs

In [5]:
def renormalise_2d(coeffs):

    Nx,Ny = coeffs.shape
    
    coeffs *= 1 / ((Nx-1) * (Ny-1))

    coeffs[0,:] *= 0.5
    coeffs[:,0] *= 0.5
    coeffs[-1,:] *= 0.5
    coeffs[:,-1] *= 0.5
    
    return coeffs

In [6]:
def renormalise_3d(coeffs):

    Nx,Ny,Nz = coeffs.shape
    
    coeffs *= 1 / ((Nx-1) * (Ny-1)* (Nz-1))

    coeffs[0,:,:] *= 0.5
    coeffs[:,0,:] *= 0.5
    coeffs[:,:,0] *= 0.5
    coeffs[-1,:,:] *= 0.5
    coeffs[:,-1,:] *= 0.5
    coeffs[:,:,-1] *= 0.5
    
    return coeffs

In [7]:
def renormalise_nd_extrema(coeffs, copy=True):
    """
    Renormalise DCT-I (norm=None) output on an extrema (Gauss-Lobatto) tensor grid
    to standard Chebyshev-series convention, for arbitrary dimension.

    This generalises:
      2D: divide by (Nx-1)(Ny-1), then halve rows/cols at indices 0 and -1
      3D: divide by (Nx-1)(Ny-1)(Nz-1), then halve each of the 6 faces, etc.

    Parameters
    ----------
    coeffs : ndarray, shape (N1, N2, ..., Nd)
        Output of scipy.fft.dctn(..., type=1, norm=None).
    copy : bool
        If True, work on a copy and return it. If False, scale in-place.

    Returns
    -------
    ndarray
        Renormalised Chebyshev coefficients (same shape as input).
    """
    arr = coeffs.copy() if copy else coeffs
    shape = arr.shape

    if any(N < 2 for N in shape):
        raise ValueError("All dimensions must have size >= 2 for DCT-I normalization.")

    # 1) divide by prod(Nr - 1)
    scale = 1.0
    for N in shape:
        scale *= (N - 1)
    arr /= scale

    # 2) halve endpoint hyperplanes along each axis
    ndim = arr.ndim
    for axis, N in enumerate(shape):
        # first index plane
        sl = [slice(None)] * ndim
        sl[axis] = 0
        arr[tuple(sl)] *= 0.5
        # last index plane
        sl[axis] = N - 1
        arr[tuple(sl)] *= 0.5

    return arr

In [8]:
# @njit
""""""
def clenshaw_3d(coeffs, x, y, z):
    """
    Evaluate a 3D Chebyshev interpolant using Clenshaw's algorithm.
    Inputs x, y, z should be in [-1, 1].
    """
    nx, ny, nz = coeffs.shape

    # Evaluate over z-axis
    tmp_z = np.zeros((nx, ny))
    for i in range(nx):
        for j in range(ny):
            tmp_z[i, j] = clenshaw_1d(coeffs[i, j, :], z)

    # Evaluate over y-axis
    tmp_y = np.zeros(nx)
    for i in range(nx):
        tmp_y[i] = clenshaw_1d(tmp_z[i, :], y)

    # Evaluate over x-axis
    return clenshaw_1d(tmp_y, x)
"""""""

SyntaxError: unterminated string literal (detected at line 23) (3122487634.py, line 23)

In [9]:
@njit
def rescale(x, a, b):
        return 2 * (x - a) / (b - a) - 1

In [10]:
@njit
def evaluate(coeffs, x, y, z, ax,bx, ay, by, az, bz):
        x_scaled = rescale(x, ax, bx)
        y_scaled = rescale(y, ay, by)
        z_scaled = rescale(z, az, bz)
        return clenshaw_3d(coeffs, x_scaled, y_scaled, z_scaled)

In [11]:
# Define grid
nx, ny, nz = 50, 50, 50
# x = np.cos(np.pi * np.arange(nx) / (nx - 1))  # Chebyshev nodes
# y = np.cos(np.pi * np.arange(ny) / (ny - 1))
# z = np.cos(np.pi * np.arange(nz) / (nz - 1))

ax, bx = -2, 1
ay, by = -1, 2
az, bz = -2, 2

x = chebyshev_nodes(nx,ax, bx)
y = chebyshev_nodes(ny,ay, by)
z = chebyshev_nodes(nz,az, bz)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Sample function
f_vals = np.sin(X) * np.cos(Y) * np.exp(Z)

# Create interpolator
coeffs = compute_chebyshev_coefficients_3d(f_vals, nx, ny, nz)

# Evaluate
x_eval = 0.5
y_eval = 0.2
z_eval = -0.3

time = perf_counter()
interp = evaluate(coeffs,x_eval, y_eval, z_eval, ax, bx, ay, by ,az, bz)
elapsed = perf_counter() - time
print(f"Time for evaluations: {elapsed *1000:.6f} ms")

print("Interpolated value:", interp)
exact = np.sin(x_eval) * np.cos(y_eval) * np.exp(z_eval)
print("Exact value:", exact)
print("Difference:", np.abs(interp - exact))


Time for evaluations: 340.898250 ms
Interpolated value: 0.3480874772327325
Exact value: 0.3480874772327324
Difference: 5.551115123125783e-17


In [12]:
x_GL = chebyshev_gauss_lobatto_nodes(nx,ax, bx)
y_GL = chebyshev_gauss_lobatto_nodes(ny,ay, by)
z_GL = chebyshev_gauss_lobatto_nodes(nz,az, bz)
X_GL, Y_GL, Z_GL = np.meshgrid(x_GL, y_GL, z_GL, indexing='ij')

# Sample function
f_vals_GL = np.sin(X_GL) * np.cos(Y_GL) * np.exp(Z_GL)

interp3D_GL = ChebyshevInterpolant3D(f_vals_GL,ax, bx,ay, by, az, bz, nodes = 'GaussLobatto')

coeffs_GL = interp3D_GL.get_coefficients()

In [13]:
coeffs_GL2 = compute_chebyshev_coefficients_3d_GL(f_vals_GL, nx, ny, nz)

In [14]:
coeffs_GL[1] ,coeffs_GL2[1]

(array([[ 1.00269848e+00,  1.39931518e+00,  6.06081781e-01, ...,
          0.00000000e+00,  1.23689239e-16,  0.00000000e+00],
        [-1.19424807e+00, -1.66663208e+00, -7.21864064e-01, ...,
         -6.18446193e-17, -2.47378477e-16,  0.00000000e+00],
        [-9.09344955e-01, -1.26903573e+00, -5.49654180e-01, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [ 1.69106381e-18,  8.12737899e-18,  6.47154911e-18, ...,
          2.22672216e-17,  3.41591358e-17,  2.12590879e-17],
        [-2.58491182e-17, -3.46998247e-17, -6.98934753e-18, ...,
          1.29672682e-17,  2.01035064e-17,  8.93848013e-18],
        [-3.32777199e-17, -5.55979757e-17, -3.79173620e-17, ...,
         -1.28961152e-17, -1.42163717e-17, -6.82465037e-18]]),
 array([[ 1.00269848e+00,  1.39931518e+00,  6.06081781e-01, ...,
          0.00000000e+00,  1.23689239e-16,  0.00000000e+00],
        [-1.19424807e+00, -1.66663208e+00, -7.21864064e-01, ...,
         -6.18446193e-17, -2.47378477e

In [15]:
interp3D = ChebyshevInterpolant3D(f_vals, ax, bx,ay, by, az, bz)

In [16]:
coeffs_GL[0,0,0]/coeffs_GL2[0,0,0]

np.float64(1.0)

In [17]:
# Test the timing of 100 evaluations using perf counter
time = perf_counter()
for _ in range(1000):
    interp = evaluate(coeffs_GL, x_eval, y_eval, z_eval, ax, bx, ay, by, az, bz)
elapsed = perf_counter() - time
print(f"Time for 1000 evaluations: {elapsed:.6f} seconds")
print(f"Average time per evaluation: {elapsed:.6f} ms")

exact = np.sin(x_eval) * np.cos(y_eval) * np.exp(z_eval)
print("Exact value:", exact)
print("Difference:", np.abs(interp - exact))


Time for 1000 evaluations: 0.242724 seconds
Average time per evaluation: 0.242724 ms
Exact value: 0.3480874772327324
Difference: 2.220446049250313e-16


In [9]:
interp3D = ChebyshevInterpolant3D(f_vals, nx,ny,nz,ax, bx,ay, by, az, bz)

In [19]:
interp3D.optimize(1e-14)

time = perf_counter()
for _ in range(1000):
    interp = interp3D_GL.evaluate(x_eval, y_eval, z_eval)
elapsed = perf_counter() - time
print(f"Time for evaluations: {elapsed:.6f} ms")

print("Interpolated value:", interp)
exact = np.sin(x_eval) * np.cos(y_eval) * np.exp(z_eval)
print("Exact value:", exact)
print("Difference:", np.abs(interp - exact))

Time for evaluations: 0.243668 ms
Interpolated value: 0.3480874772327322
Exact value: 0.3480874772327324
Difference: 2.220446049250313e-16


In [21]:
interp3D.optimize(1e-14)

interp3D.precompute_x(x_eval)

time = perf_counter()
for _ in range(1000):
    interp = interp3D.evaluate_yz( y_eval, z_eval)
elapsed = perf_counter() - time
print(f"Time for evaluations: {elapsed:.6f} ms")

print("Interpolated value:", interp)
exact = np.sin(x_eval) * np.cos(y_eval) * np.exp(z_eval)
print("Exact value:", exact)
print("Difference:", np.abs(interp - exact))

Time for evaluations: 0.004980 ms
Interpolated value: 0.34808747723273237
Exact value: 0.3480874772327324
Difference: 5.551115123125783e-17


In [29]:
interp3D.optimize(1e-12)
interp3D.precompute_x(x_eval)

time = perf_counter()
interp = interp3D.evaluate_yz(y_eval, z_eval)
elapsed = perf_counter() - time
print(f"Time for evaluations: {elapsed:.6f} ms")

print("Interpolated value:", interp)
exact = np.sin(x_eval) * np.cos(y_eval) * np.exp(z_eval)
print("Exact value:", exact)
print("Difference:", np.abs(interp - exact))

Time for evaluations: 0.064834 ms
Interpolated value: 0.348087477232769
Exact value: 0.3480874772327324
Difference: 3.658184866139891e-14


So a little on the slow side

## Pruning Chebyshev Coefficients

In [12]:
def prune_trailing_slices_greedy(coeffs, delta=1e-12, scale='max'):
    """
    Structured pruning of Chebyshev coeffs (nx,ny,nz) by removing *trailing* slices
    along axes to reduce Clenshaw lengths while controlling a global relative error.

    Error control: sum(|discarded coeffs|) <= delta * S, with S given by `scale`.

    Parameters
    ----------
    coeffs : (nx,ny,nz) ndarray
        Chebyshev coefficients c[i,j,k].
    delta : float
        Allowed global relative error.
    scale : {'max','l1','l2', float}
        Scale S for the relative tolerance:
          - 'max' (default): S = max(abs(coeffs))
          - 'l1' : S = sum(abs(coeffs))
          - 'l2' : S = norm(coeffs.ravel(), 2)
          - float: use the provided value

    Returns
    -------
    kmax : tuple (nx_keep, ny_keep, nz_keep)
        Truncation limits; keep block coeffs[:nx_keep, :ny_keep, :nz_keep].
    discarded_abs_sum : float
        Sum of |coeffs| discarded (an upper bound on absolute uniform error).
    """
    c = np.asarray(coeffs)
    nx, ny, nz = c.shape
    if c.size == 0:
        return (0,0,0), 0.0

    abs_c = np.abs(c)
    if scale == 'max':
        S = abs_c.max()
    elif scale == 'l1':
        S = abs_c.sum()
    elif scale == 'l2':
        S = np.sqrt(np.sum(abs_c**2))
    elif isinstance(scale, (int, float)):
        S = float(scale)
    else:
        raise ValueError("scale must be 'max', 'l1', 'l2', or a float")

    if S == 0.0:
        # all-zero; keep nothing
        return (0,0,0), 0.0

    tau = delta * S

    # Precompute L1 costs of each trailing slice along each axis
    # cost_x[i] = sum over j,k of |c[i,j,k]|
    cost_x = abs_c.sum(axis=(1,2))
    cost_y = abs_c.sum(axis=(0,2))
    cost_z = abs_c.sum(axis=(0,1))

    # Start from full sizes; greedily trim the cheapest next trailing slice
    ix, iy, iz = nx, ny, nz
    discarded = 0.0

    # Next slice costs (look at the slice right before the current bound)
    def next_costs(ix, iy, iz):
        cx = cost_x[ix-1] if ix > 0 else np.inf
        cy = cost_y[iy-1] if iy > 0 else np.inf
        cz = cost_z[iz-1] if iz > 0 else np.inf
        return cx, cy, cz

    while True:
        cx, cy, cz = next_costs(ix, iy, iz)
        # pick cheapest axis
        axis = np.argmin([cx, cy, cz])
        inc = [cx, cy, cz][axis]

        # if any axis already at 0, its cost is inf; if all inf, break
        if not np.isfinite(inc):
            break
        if discarded + inc > tau:
            # next trim would exceed budget
            break

        discarded += inc
        if axis == 0:
            ix -= 1
        elif axis == 1:
            iy -= 1
        else:
            iz -= 1

        # stop if everything trimmed
        if ix == 0 or iy == 0 or iz == 0:
            break

    return (ix, iy, iz), discarded

In [13]:
def trim_to_block(coeffs, kmax):
    ix, iy, iz = kmax
    return coeffs[:ix, :iy, :iz].copy()

def block_to_coo(coeff_block):
    """
    Return sparse COO-like representation of a dense block (indices, values).
    Useful if you later want sparse evaluation (usually slower than Clenshaw).
    """
    nz_mask = coeff_block != 0
    idx = np.argwhere(nz_mask)           # shape (nnz, 3)
    vals = coeff_block[nz_mask].ravel()  # shape (nnz,)
    return idx, vals

In [14]:
def clenshaw_1d_T(coeffs_1d, t):
    # Standard Chebyshev Clenshaw on t in [-1,1]
    b1 = 0.0
    b2 = 0.0
    tt = 2.0 * t
    for ck in coeffs_1d[::-1]:
        b0 = tt * b1 - b2 + ck
        b2 = b1
        b1 = b0
    return 0.5 * (b1 - b2)

@njit
def clenshaw_3d_with_kmax(coeffs, x, y, z, ax, bx, ay, by, az, bz, kmax=None):
    """
    coeffs: (nx,ny,nz) full coefficients (you may pass a trimmed block)
    kmax: (ix,iy,iz) optional; if None uses coeffs.shape
    """
    nx, ny, nz = coeffs.shape
    if kmax is None:
        ix, iy, iz = nx, ny, nz
    else:
        ix, iy, iz = kmax

    # Scale x,y,z to [-1,1]
    tx = (2*x - (bx+ax)) / (bx-ax)
    ty = (2*y - (by+ay)) / (by-ay)
    tz = (2*z - (bz+az)) / (bz-az)

    # Evaluate along z first using only [:,:iz]
    tmp_yx = np.empty((ix, iy))
    for i in range(ix):
        for j in range(iy):
            tmp_yx[i, j] = clenshaw_1d(coeffs[i, j, :iz], tz)

    # Then along y using only [:iy]
    tmp_x = np.empty(ix)
    for i in range(ix):
        tmp_x[i] = clenshaw_1d(tmp_yx[i, :iy], ty)

    # Finally along x using only [:ix]
    return clenshaw_1d(tmp_x[:ix], tx)

In [15]:
# 1) Compute full coeffs (nx,ny,nz)
# 2) Prune to meet delta
kmax, discarded = prune_trailing_slices_greedy(coeffs, delta=1e-6, scale='max')

# 3a) Fast path: just pass kmax to Clenshaw (no copy needed)
val = clenshaw_3d_with_kmax(coeffs, x_eval, y_eval, z_eval, ax, bx, ay, by, az, bz, kmax=kmax)

# 3b) Or physically trim to a smaller tensor to reduce memory/cache footprint
coeffs_trim = trim_to_block(coeffs, kmax)
val2 = clenshaw_3d_with_kmax(coeffs_trim, x_eval, y_eval, z_eval, ax, bx, ay, by, az, bz)

In [16]:
[exact, val, val2, exact-val,exact-val2]

[np.float64(0.3480874772327324),
 0.34808746019705605,
 0.34808746019705605,
 np.float64(1.703567636957004e-08),
 np.float64(1.703567636957004e-08)]

In [17]:
def prune_coeffs_3d(coeffs, delta):
    """
    Prune 3D Chebyshev coefficients by estimating required truncation
    depth along z for each (i,j) slice.

    Parameters
    ----------
    coeffs : np.ndarray, shape (Nx, Ny, Nz)
        Chebyshev coefficients.
    delta : float
        Target relative tolerance.

    Returns
    -------
    NzMaxes : np.ndarray, shape (Nx, Ny), dtype=int
        Truncation depth in z for each (i,j).
    NyMaxes : np.ndarray, shape (Nx,), dtype=int
        Truncation depth in y for each i.
    NxMax  : int
        Truncation depth in x.
    """
    Nx, Ny, Nz = coeffs.shape

    # global scaling reference
    max_c = np.max(np.abs(coeffs))
    threshold = delta * max_c

    # per-(i,j) z-truncations
    NzMaxes = np.zeros((Nx, Ny), dtype=np.int64)
    for i in range(Nx):
        for j in range(Ny):
            col = np.abs(coeffs[i, j, :])
            nz_max = 0
            for k in range(Nz-1, -1, -1):
                if col[k] > threshold / (Nx + Ny + Nz):
                    nz_max = k + 1
                    break
            NzMaxes[i, j] = nz_max

    # per-i y-truncations
    NyMaxes = np.zeros(Nx, dtype=np.int64)
    for i in range(Nx):
        ny_max = 0
        for j in range(Ny-1, -1, -1):
            if NzMaxes[i, j] > 0:
                ny_max = j + 1
                break
        NyMaxes[i] = ny_max

    # x-truncation
    NxMax = 0
    for i in range(Nx-1, -1, -1):
        if NyMaxes[i] > 0:
            NxMax = i + 1
            break

    return NzMaxes, NyMaxes, NxMax


@njit
def clenshaw_3d_NzMaxes(coeffs, x, y, z, NzMaxes, NyMaxes, NxMax, 
                ax=-1., bx=1., ay=-1., by=1., az=-1., bz=1.):
    """
    Evaluate a 3D Chebyshev interpolant using Clenshaw with slice truncation.
    """
    # Map x,y,z into [-1,1]
    tx = (2 * x - (bx + ax)) / (bx - ax)
    ty = (2 * y - (by + ay)) / (by - ay)
    tz = (2 * z - (bz + az)) / (bz - az)

    # Step 1: evaluate along z for each (i,j)
    tmp_yz = np.empty((NxMax, NyMaxes.max()))
    for i in range(NxMax):
        for j in range(NyMaxes[i]):
            nz_max = NzMaxes[i, j]
            row = np.array(coeffs[i, j, :], dtype=np.float64, copy=True)
            tmp_yz[i, j] = clenshaw_1d(row, tz, int(nz_max))

    # Step 2: collapse along y for each i
    tmp_x = np.empty(NxMax)
    for i in range(NxMax):
        tmp_x[i] = clenshaw_1d(tmp_yz[i, :NyMaxes[i]], ty, NyMaxes[i])

    # Step 3: collapse along x
    return clenshaw_1d(tmp_x, tx, NxMax) 



In [19]:
interp3D.optimize(rel_tol=1e-6)
time = perf_counter()
val3 = interp3D.evaluate(x_eval, y_eval, z_eval)
elapsed = perf_counter() - time
print(f"Time for evaluations: {elapsed *1000:.6f} ms")
val3, exact - val3

Time for evaluations: 0.063333 ms


(0.12865547823362733, np.float64(0.2194319989991051))

In [20]:
interp3D.effective_grid(), kmax

((10, 10, 11), (10, 10, 11))

In [21]:
# Test the timing of 100 evaluations using perf counter
time = perf_counter()
for _ in range(1000):
    clenshaw_3d(coeffs, x_eval, y_eval, z_eval, ax, bx, ay, by, az, bz,kmax=kmax)

elapsed = perf_counter() - time
print(f"Time for 1000 evaluations: {elapsed:.6f} seconds")
print(f"Average time per evaluation: {elapsed:.6f} ms")

Time for 1000 evaluations: 0.001892 seconds
Average time per evaluation: 0.001892 ms


In [22]:
# Test the timing of 100 evaluations using perf counter
time = perf_counter()
for _ in range(1000):
    interp3D.evaluate(x_eval, y_eval, z_eval)

elapsed = perf_counter() - time
print(f"Time for 1000 evaluations: {elapsed:.6f} seconds")
print(f"Average time per evaluation: {elapsed:.6f} ms")

Time for 1000 evaluations: 0.002425 seconds
Average time per evaluation: 0.002425 ms


## Trying out the tucker decomposition

In [22]:
import tensorly as tl 
from tensorly.decomposition import tucker, parafac



In [256]:
core, factors = tucker(coeffs, rank=[1, 1, 1])

In [257]:
factors[0].shape, factors[1].shape, factors[2].shape

((50, 1), (40, 1), (60, 1))

In [258]:
core[0, 0, 0]  # Should be close to the value at (0, 0, 0) in coeffs

np.float64(-3.5169303523773516)

In [261]:
@njit
def tucker_contract(core, a, b, c):
    val = 0.0
    P, Q, R = core.shape
    for p in range(P):
        for q in range(Q):
            for r in range(R):
                val += core[p,q,r] * a[p] * b[q] * c[r]
    return val

@njit
def tucker_evaluation(x, y, z, core, factors,ax, bx, ay, by, az, bz):

    x_scaled = rescale(x, ax, bx)
    y_scaled = rescale(y, ay, by)
    z_scaled = rescale(z, az, bz)
    # For each column of the factor matrices, compute projection using Clenshaw
    a = np.array([clenshaw_1d( factors[0][:,p],x_scaled) for p in range(factors[0].shape[1])])
    b = np.array([clenshaw_1d(factors[1][:,q],y_scaled) for q in range(factors[1].shape[1])])
    c = np.array([clenshaw_1d(factors[2][:,r],z_scaled) for r in range(factors[2].shape[1])])

    # Contract with core tensor
    # val = tl.tenalg.multi_mode_dot(core, [a, b, c], modes=[0,1,2])
    val = tucker_contract(core, a, b, c)

    return val

In [263]:
time = perf_counter()
for _ in range(100):
    tucker_evaluation(x_eval, y_eval, z_eval, core, factors,ax, bx, ay, by, az, bz)
elapsed = perf_counter() - time
print(f"Time for 100 evaluations: {elapsed:.6f} seconds")
print(f"Average time per evaluation: {1000*elapsed / 100:.6f} ms")

tucker_evaluation(x_eval, y_eval, z_eval, core, factors,ax, bx, ay, by, az, bz) - exact

Time for 100 evaluations: 0.002509 seconds
Average time per evaluation: 0.025090 ms


np.float64(4.996003610813204e-16)

In [226]:
# CP decomposition (rank-R approximation)
cp_rank = 5
weights, factors = parafac(coeffs, rank=cp_rank)

U1, U2, U3 = factors

len(U1), len(U2), len(U3)

(50, 40, 60)

In [227]:
# Evaluation with CP decomposition
def cp_evaluation(x, y, z, weights, factors,ax, bx, ay, by, az, bz):
    U1, U2, U3 = factors
    R = len(weights)
    x_scaled = rescale(x, ax, bx)
    y_scaled = rescale(y, ay, by)
    z_scaled = rescale(z, az, bz)

    val = 0.0
    for r in range(R):
        a = clenshaw_1d(U1[:,r], x_scaled)
        b = clenshaw_1d(U2[:,r],y_scaled)
        c = clenshaw_1d(U3[:,r],z_scaled)
        val += weights[r] * a * b * c
    return val

In [228]:
cp_evaluation(x_eval, y_eval, z_eval, weights, factors,ax, bx, ay, by, az, bz) - exact

np.float64(5.051514762044462e-15)

In [229]:
time = perf_counter()
for _ in range(1000):
    cp_evaluation(x_eval, y_eval, z_eval, weights, factors,ax, bx, ay, by, az, bz)
elapsed = perf_counter() - time
print(f"Time for 1000 evaluations: {elapsed:.6f} seconds")
print(f"Average time per evaluation: {elapsed:.6f} ms")

cp_evaluation(x_eval, y_eval, z_eval, weights, factors,ax, bx, ay, by, az, bz) - exact

Time for 1000 evaluations: 0.271577 seconds
Average time per evaluation: 0.271577 ms


np.float64(5.051514762044462e-15)

In [232]:
def truncate_tail_l1(vec, alpha):
    """Return a truncated copy of vec where the trailing tail has ℓ1 mass ≤ alpha * ||vec||_1."""
    v = np.asarray(vec, dtype=float)
    total = np.sum(np.abs(v))
    if total == 0.0:
        return v[:1]  # keep T_0 only
    budget = alpha * total
    # Walk from the end, removing until tail_l1 ≤ budget
    tail = 0.0
    m = len(v) - 1
    while m >= 0 and tail + abs(v[m]) <= budget:
        tail += abs(v[m])
        m -= 1
    # Keep indices 0..m inclusively
    keep_len = max(m + 1, 1)  # keep at least T_0
    return v[:keep_len]

def prune_cp_vectors(weights, factors, delta=1e-3, alpha_equal=True, alphas=None):
    """
    Per-component, per-mode tail truncation with provable sup-norm relative error ≤ delta.
    For each rank-1 term, enforce tail budgets alpha_x, alpha_y, alpha_z.

    Returns:
      weights_out: shape (R_kept,)
      U_list, V_list, W_list: lists of 1D arrays (variable lengths after truncation)
    """
    U, V, W = factors
    R = len(weights)

    # Choose per-mode budgets
    if alpha_equal:
        alpha = delta / 4.0  # safe default
        alpha_x = alpha_y = alpha_z = alpha
    else:
        assert alphas is not None and len(alphas) == 3
        alpha_x, alpha_y, alpha_z = alphas

    weights_out = []
    U_list, V_list, W_list = [], [], []

    for r in range(R):
        u = U[:, r]; v = V[:, r]; w = W[:, r]

        # Truncate tails to meet ℓ1 budgets
        u_t = truncate_tail_l1(u, alpha_x)
        v_t = truncate_tail_l1(v, alpha_y)
        w_t = truncate_tail_l1(w, alpha_z)

        # Skip entirely negligible terms if any factor collapses to ~zero
        if (np.allclose(u_t, 0) or np.allclose(v_t, 0) or np.allclose(w_t, 0)
            or abs(weights[r]) == 0.0):
            continue

        weights_out.append(weights[r])
        U_list.append(u_t); V_list.append(v_t); W_list.append(w_t)

    weights_out = np.array(weights_out)
    return weights_out, U_list, V_list, W_list

def chebyshev_interp_cp_pruned(x, y, z, weights, U_list, V_list, W_list,ax, bx, ay, by, az, bz):
    """Evaluate CP after per-vector tail pruning (variable-length factors)."""
    val = 0.0
    x_rescaled = rescale(x, ax, bx)
    y_rescaled = rescale(y, ay, by)
    z_rescaled = rescale(z, az, bz)
    for lam, u, v, w in zip(weights, U_list, V_list, W_list):
        a = clenshaw_1d(u, x_rescaled)
        b = clenshaw_1d(v, y_rescaled)
        c = clenshaw_1d(w,z_rescaled)
        val += lam * a * b * c
    return val


In [236]:
delta = 1e-2  # target relative sup error per CP term
weights_p, U_list, V_list, W_list = prune_cp_vectors(weights, factors, delta=delta)

fx = chebyshev_interp_cp_pruned(x_eval, y_eval, z_eval, weights_p, U_list, V_list, W_list,ax, bx, ay, by, az, bz)
(fx - exact)/exact

np.float64(0.003606040177041673)

In [148]:
len(U_list), len(V_list), len(W_list), U_list[0].shape, V_list[0].shape, W_list[0].shape

(1, 1, 1, (10,), (9,), (10,))

In [149]:
time = perf_counter()
for _ in range(1000):
    chebyshev_interp_cp_pruned(x_eval, y_eval, z_eval, weights_p, U_list, V_list, W_list)
elapsed = perf_counter() - time
print(f"Time for 100 evaluations: {elapsed:.6f} seconds")
print(f"Average time per evaluation: {1000*elapsed / 1000:.6f} ms")

Time for 100 evaluations: 0.067450 seconds
Average time per evaluation: 0.067450 ms


## Test against a spline

In [157]:
from multispline.spline import TricubicSpline

xs= np.linspace(ax, bx, 2**5 +1)
ys = np.linspace(ay, by, 2**5 +1)
zs = np.linspace(az, bz, 2**5 +1)

f_vals_ES = np.zeros((len(xs), len(ys), len(zs)))
for i, x in enumerate(xs):
    for j, y in enumerate(ys):
        for k, z in enumerate(zs):
            f_vals_ES[i, j, k] = np.sin(x) * np.cos(y) * np.exp(z)


spline = TricubicSpline(xs, ys, zs, f_vals_ES)

In [159]:
time = perf_counter()
for _ in range(1000):
    interp_spline = spline(x_eval, y_eval, z_eval)

elapsed = perf_counter() - time
print(f"Time for 1000 evaluations with spline: {elapsed:.6f} seconds")
print(f"Average time per evaluation with spline: {1000*elapsed / 1000:.6f} ms")

Time for 1000 evaluations with spline: 0.000538 seconds
Average time per evaluation with spline: 0.000538 ms


In [160]:
spline(x_eval,y_eval,z_eval) - exact

np.float64(0.6315480115296384)

## Now Check relative error across parameter space

In [173]:
cheb_error = np.zeros((len(xs), len(ys), len(zs)))

delta = 1e-12  # target relative sup error per CP term
weights_p, U_list, V_list, W_list = prune_cp_vectors(weights, factors, delta=delta)

time = perf_counter()
for i, x in enumerate(xs):
    for j, y in enumerate(ys):
        for k, z in enumerate(zs):
            exact = np.sin(x) * np.cos(y) * np.exp(z)
            p = chebyshev_interp_cp_pruned(x, y, z, weights_p, U_list, V_list, W_list)
            cheb_error[i,j,k] = np.abs((p - exact))
elapsed = perf_counter() - time
print(f"Average time per evaluation: {1000*elapsed / (len(xs)* len(ys)*len(zs)):.6f} ms")

# Return max error
max_error = np.max(np.abs(cheb_error))
print(f"Maximum error in Chebyshev interpolation: {max_error}")
# Return Median error
median_error = np.median(np.abs(cheb_error))
print(f"Median error in Chebyshev interpolation: {median_error}")

Average time per evaluation: 0.055519 ms
Maximum error in Chebyshev interpolation: 3.7320146972774637e-13
Median error in Chebyshev interpolation: 6.0285110237146e-14


In [170]:
len(W_list)

1

In [167]:
U_list[0].shape, V_list[0].shape, W_list[0].shape

((8,), (7,), (8,))

In [165]:
x_nodes = chebyshev_nodes(nx,ax, bx)
y_nodes = chebyshev_nodes(ny,ay, by)
z_nodes = chebyshev_nodes(nz,az, bz)

spline_error = np.zeros((len(x_nodes), len(y_nodes), len(z_nodes)))

for i, x in enumerate(x_nodes):
    for j, y in enumerate(y_nodes):
        for k, z in enumerate(z_nodes):
            exact = np.sin(x) * np.cos(y) * np.exp(z)
            p = spline(x, y, z)
            spline_error[i,j,k] = np.abs((p - exact))


# Return max error
max_error = np.max(np.abs(spline_error))
print(f"Maximum error in Chebyshev interpolation: {max_error}")
# Return Median error
median_error = np.median(np.abs(spline_error))
print(f"Median error in Chebyshev interpolation: {median_error}")

Maximum error in Chebyshev interpolation: 9.258830968583709e-07
Median error in Chebyshev interpolation: 2.5022902416793613e-08


In [162]:
# return any values of f_vals_ES that are zero
zero_f_vals = f_vals_ES[np.isclose(f_vals_ES, 0)]
print(f"Number of zero values in f_vals_ES: {len(zero_f_vals)}")

Number of zero values in f_vals_ES: 1089


So this CP decomposition + pruning is doing pretty well. 
Need to now start interpolating equatorial Kerr fluxes to make sure its fast enough. Undeniably this is very memory efficient!

I wonder if our bottleneck is now just the conversion from p to u, at which point we might be okay since that effects both equally.

By the end of this week, I would like to achieve the following:

- Test 3D implimentation against spline in FEW and report speed
- Add in 3D optimisation
- Test tensor decomposition
- Add in functionality to precompute over a and test timing
- Finally add in basic 4D implimentation, ask Zach for a 4D spline